In [1]:
import pandas as pd
import csv

with open('/content/nutrisi_clean.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()

header = [h.strip().lower() for h in lines[0].strip().split(',')]
clean_rows = []
for line in lines[1:]:
    l = line.strip()
    if l.startswith('"') and l.endswith('"'): l = l[1:-1]
    l = l.replace('""', '"')
    reader = csv.reader([l], delimiter=',', quotechar='"')
    try:
        row = next(reader)
        if len(row) >= len(header): clean_rows.append(row[:len(header)])
        else:
            row.extend([None] * (len(header) - len(row)))
            clean_rows.append(row)
    except: continue

df = pd.DataFrame(clean_rows, columns=header)
df.columns = df.columns.str.strip().str.lower()

def clean_simple(val):
    if val is None or val == '' or val == '-': return 0.0
    s = str(val).replace(',', '.')
    try:
        return float(s)
    except:
        return val

df = df.map(clean_simple)

cols_to_check = ['protein', 'lemak', 'karbohidrat', 'natrium  (na)', 'kalium  (ka)', 'fosfor  (p)', 'energi']

for col in df.columns:
    if any(target in col for target in cols_to_check):
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

        if 'protein' in col:
            df.loc[df[col] >= 20, col] = df[col] / 10.0

        if 'kalium' in col:
            df.loc[df[col] >= 100, col] = df[col] / 10.0

        if 'energi' in col:
            df.loc[df[col] >= 500, col] = df[col] / 10.0

nasi_fix = df[df['makanan'].astype(str).str.contains('Nasi', na=False, case=False)].head(5)
cols_show = [c for c in df.columns if any(x in c for x in ['makan', 'protein', 'kalium', 'energi'])]
print(nasi_fix[cols_show].to_string(index=False))

df.to_csv('nutrisi_recovered.csv', index=False)

         makanan  energi  protein  kalium  (ka) kelompok makanan
            Nasi   180.0      3.0          38.0         Serealia
        Nasi tim   120.0      2.4          23.9         Serealia
Nasi beras merah   149.0      2.8          91.4         Serealia
     Nasi jagung   357.0      8.8          30.4         Serealia
      Nasi gemuk   192.0      3.8          63.0         Serealia


In [2]:
!pip install -q langchain langchain-community langchain-huggingface langchain-google-genai chromadb sentence-transformers pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 

Setup Vector Database (ChromaDB) & Embedding Knowledge Base

In [3]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

def build_rag_desc(row):
    return (f"{row['makanan']} ({row['kelompok makanan']}) mengandung "
            f"energi {row['energi']} kkal, protein {row['protein']}g, "
            f"natrium {row['natrium  (na)']}mg, kalium {row['kalium  (ka)']}mg, "
            f"dan fosfor {row['fosfor  (p)']}mg.")

df['deskripsi_rag'] = df.apply(build_rag_desc, axis=1)

# Document Transformation & Metadata Enrichment
documents = []
for _, row in df.iterrows():
    # Masukkan angka ke metadata untuk kebutuhan filter klinis nanti
    metadata = {
        "makanan": str(row['makanan']),
        "kelompok": str(row['kelompok makanan']),
        "natrium": float(row['natrium  (na)']),
        "kalium": float(row['kalium  (ka)']),
        "fosfor": float(row['fosfor  (p)']),
        "protein": float(row['protein'])
    }
    documents.append(Document(page_content=row['deskripsi_rag'], metadata=metadata))

# Pemilihan Model Embedding & Vectorization
# Menggunakan model multilingual
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

print(f"{len(documents)} dokumen siap dimasukkan ke database vektor.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

1151 dokumen siap dimasukkan ke database vektor.


In [4]:
from langchain_community.vectorstores import Chroma

# Storage & Persistency
persist_directory = "db_pangan_fix"

# Membuat database dan menyimpannya ke folder lokal
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Database berhasil disimpan secara permanen di folder: '{persist_directory}'")

/tmp/ipykernel_1075/875716390.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Database berhasil disimpan secara permanen di folder: 'db_pangan_fix'


Implementasi Retrieval Pipeline

In [5]:
# k=10 artinya kita mengambil 10 referensi makanan terdekat
retriever = vector_db.as_retriever(search_kwargs={"k": 10})

print("Retriever berhasil dikonfigurasi")

Retriever berhasil dikonfigurasi


In [6]:
def format_docs(docs):
    # Menggabungkan hasil pencarian menjadi satu blok teks untuk LLM
    return "\n\n".join(doc.page_content for doc in docs)

# Uji coba pencarian semantik
uji_cari = retriever.invoke("Saran sumber karbohidrat")

print(f"Hasil Testing Semantic Search untuk '{uji_cari}':")
for d in uji_cari:
    print(f"- {d.page_content}")

Hasil Testing Semantic Search untuk '[Document(metadata={'natrium': 64.0, 'kalium': 242.1, 'makanan': 'Domba, daging, gemuk, segar', 'fosfor': 157.0, 'kelompok': 'Daging', 'protein': 15.7}, page_content='Domba, daging, gemuk, segar (Daging) mengandung energi 317.0 kkal, protein 15.7g, natrium 64.0mg, kalium 242.1mg, dan fosfor 157.0mg.'), Document(metadata={'natrium': 500.0, 'protein': 49.6, 'fosfor': 130.0, 'kelompok': 'Daging', 'makanan': 'Buaya, daging, dendeng, mentah', 'kalium': 680.0}, page_content='Buaya, daging, dendeng, mentah (Daging) mengandung energi 365.0 kkal, protein 49.6g, natrium 500.0mg, kalium 680.0mg, dan fosfor 130.0mg.'), Document(metadata={'protein': 18.8, 'kelompok': 'Daging', 'kalium': 378.0, 'fosfor': 170.0, 'natrium': 105.0, 'makanan': 'Sapi, daging, lemak sedang, segar'}, page_content='Sapi, daging, lemak sedang, segar (Daging) mengandung energi 201.0 kkal, protein 18.8g, natrium 105.0mg, kalium 378.0mg, dan fosfor 170.0mg.'), Document(metadata={'natrium': 6

Integrasi LLM API

In [8]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "AIzaSyDQOfaU6U4HDNociL8HjGI49---"

try:
    llm = ChatGoogleGenerativeAI(
        model="gemini-3-flash-preview",
        temperature=0.1
    )
    test_res = llm.invoke("Halo Dr. Ginjal!")
    print("Gemini 3 siap")
except Exception as e:
    print(f"Kendala: {e}")
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

Gemini 3 siap


In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Kerangka RAG awal (belum pakai prompt safety)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm
    | StrOutputParser()
)

Prompt Engineering (Medical Safety)

In [10]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt khusus diet Ginjal
template = """
Kamu adalah Asisten Gizi Klinis untuk Penyakit Ginjal Kronis (CKD).

DATA REFERENSI:
{context}

PERTANYAAN PASIEN: {question}

INSTRUKSI:
1. Jawab berdasarkan DATA REFERENSI saja.
2. Jika Kalium atau Natrium > 200mg, WAJIB beri peringatan pembatasan.
3. Selalu akhiri dengan: "Informasi ini adalah edukasi, harap hubungi dokter Anda."

JAWABAN:"""

prompt = ChatPromptTemplate.from_template(template)

# Update RAG Chain dengan Prompt Safety
final_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [11]:
pertanyaan_final = "Berapa protein dalam nasi putih? Bolehkah saya makan banyak?"
jawaban = final_rag_chain.invoke(pertanyaan_final)

print(jawaban.replace('**', ''))

Berdasarkan data referensi yang tersedia, berikut adalah kandungan protein pada beberapa jenis olahan nasi (karena "nasi putih" polos tidak tertera secara spesifik, berikut adalah pilihan yang tersedia):

*   Nasi tim: 2,4 g protein.
*   Nasi minyak: 3,5 g protein.
*   Nasi gemuk: 3,8 g protein.
*   Nasi rames: 10,3 g protein.
*   Pundut nasi: 19,0 g protein.

Bolehkah makan banyak?
Bagi pasien Penyakit Ginjal Kronis (CKD), asupan protein, natrium, dan kalium harus dikontrol dengan ketat sesuai anjuran dokter atau ahli gizi. Mengonsumsi dalam jumlah banyak tidak disarankan, terutama pada jenis olahan nasi yang memiliki kadar protein atau natrium yang tinggi karena dapat memperberat kerja ginjal.

Peringatan Pembatasan:
Berdasarkan data referensi, beberapa bahan berikut mengandung Natrium atau Kalium lebih dari 200mg sehingga wajib dibatasi:
*   Nasi gemuk: Mengandung Natrium 355,0 mg.
*   Nasi rames: Mengandung Natrium 265,0 mg.
*   Ikan Teri nasi (kering, mentah): Mengandung Natrium 3

In [12]:
pertanyaan_final = "Berapa nutrisi dalam tepung beras mentah? Aman tidak kalau saya makan ini?"
jawaban = final_rag_chain.invoke(pertanyaan_final)

print(jawaban.replace('**', ''))

Berdasarkan data referensi, kandungan nutrisi dalam tepung beras mentah adalah sebagai berikut:
*   Energi: 353.0 kkal
*   Protein: 7.0 g
*   Natrium: 5.0 mg
*   Kalium: 241.0 mg
*   Fosfor: 140.0 mg

Peringatan: Karena kandungan Kalium pada tepung beras mentah sebesar 241.0 mg (melebihi ambang batas 200 mg), maka konsumsi bahan makanan ini perlu dibatasi bagi penderita Penyakit Ginjal Kronis.

Informasi ini adalah edukasi, harap hubungi dokter Anda.


In [13]:
pertanyaan_final = "Sebutkan protein dan natrium dalam nasi gemuk. Bolehkah pasien ginjal makan ini?"
jawaban = final_rag_chain.invoke(pertanyaan_final)

print(jawaban.replace('**', ''))

Berdasarkan data referensi yang tersedia, berikut adalah informasi mengenai kandungan gizi dalam nasi gemuk:

*   Protein: 3,8 g
*   Natrium: 355,0 mg

Peringatan Pembatasan:
Nasi gemuk mengandung natrium sebesar 355,0 mg. Karena kandungan natriumnya melebihi 200 mg, konsumsi makanan ini wajib dibatasi bagi pasien ginjal untuk menghindari penumpukan cairan dan tekanan darah tinggi.

Informasi ini adalah edukasi, harap hubungi dokter Anda.


In [15]:
daftar_pertanyaan = [
    "Berapa protein dalam nasi jagung?",
    "Apakah bihun goreng instan aman untuk pasien ginjal?",
    "Sebutkan kalium dalam pundut nasi."
]

for tanya in daftar_pertanyaan:
    print(f"Pertanyaan: {tanya}")
    hasil = final_rag_chain.invoke(tanya)
    print(hasil.replace('**', ''))

Pertanyaan: Berapa protein dalam nasi jagung?
Berdasarkan data referensi, nasi jagung mengandung protein sebesar 8.8g. Kandungan natrium pada nasi jagung adalah 2.0mg dan kalium sebesar 30.4mg.

Informasi ini adalah edukasi, harap hubungi dokter Anda.
Pertanyaan: Apakah bihun goreng instan aman untuk pasien ginjal?
Berdasarkan data referensi, bihun goreng instan mengandung:
*   Energi: 381.0 kkal
*   Protein: 6.1g
*   Natrium: 928.0mg
*   Kalium: 0.0mg
*   Fosfor: 151.0mg

Peringatan Pembatasan:
Bihun goreng instan mengandung natrium sebesar 928.0mg. Karena kandungan natrium tersebut jauh melebihi 200mg, konsumsi makanan ini wajib dibatasi bagi pasien ginjal kronis untuk mencegah komplikasi seperti tekanan darah tinggi dan penumpukan cairan.

Informasi ini adalah edukasi, harap hubungi dokter Anda.
Pertanyaan: Sebutkan kalium dalam pundut nasi.
Berdasarkan data referensi, kandungan kalium dalam pundut nasi adalah 174.0 mg.

Informasi ini adalah edukasi, harap hubungi dokter Anda.
